<center><h1 style="margin-bottom: 0px;">Artificial Intelligence (25/26)</h1></center>
<center><h2 style="margin-top: 0px;">Monte Carlo Tree Search Implementation </h2></center>

#### <center> Joana Antunes (202405702), Sílvia Pinto (202405988) </center> <br>

### **4. Monte Carlo Tree Search** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;"> In this notebook, we create various versions of the Monte Carlo Tree Search (MCTS) algorithm. It is based on the execution of multiple random simulations (playouts) starting from a given state of the game. For each possible move:</div>

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**1.** *That move is applied to the current game state;*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**2.** *From that point on, the game is simulated in a random way;*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**3.** *The final result of that random simulation is registered (victory, defeat or draw);*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**4.** *The process described in points 2 and 3 is repeated multiple times, and the mean of the results is calculated.*

<div style="text-align: justify;"><p style="text-indent: 2em;">The chosen move is the one with the best mean result.</div>

#### **4.1. Standard Monte Carlo with UCT** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">Now, we implement the standard Monte Carlo Tree Search algorithm with UCT(Upper Confidence bound for Trees). MCTS builds a search tree incrementally and focuses its simulations on the most promising moves. The UCT formula used to select the best child balances exploitation (choosing moves that have performed well) and exploration (trying moves that have been visited less):

$$UCT = \frac{w}{n} + c \cdot \sqrt{\frac{\ln N}{n}}$$

Where $w$ is the number of wins, $n$ is the number of visits to this node, $N$ is the total visits to the parent, and $c$ is the exploration constant (typically $\sqrt{2} \approx 1.41$).
<p style="text-indent: 2em;">The core stucture of this version is MCTSNode class, which represents a node in the search tree. Each node corresponds to the current game state, the move that led to it, a list of child nodes and two key statistics used by the UCT formula: the number of visits and the accumulated score. The best_child method implements the UCT formula, which balances between choosing moves that have performed well and trying moves that have been visited less. The roll_out function simulates the rest of the game from a given node.</div>

In [2]:
import math
import random

class MCTSNode:
    def __init__(self, state, parent=None, move=None):
        self.state = state                                          # Current game state.
        self.parent = parent                                        # Parent node in the tree.
        self.move = move                                            # Move that led to this state.
        self.children = []                                          # List of child nodes.
        self.untried_moves = state.get_valid_moves()                # Moves not yet explored.
        self.visits = 0                                             # Number of times node was visited.
        self.score = 0.0                                            # Total reward accumulated.
        self.move_player = "X" if state.cur_player == "O" else "O"  # Player who made the move that led to this node.

    def fully_expanded(self):                                       # Check if all possible moves from this node have been explored.
        return len(self.untried_moves) == 0
    
    def best_child(self, exp = 1.41):                               # Select the best child using UCT.
        best_score = -float("inf")
        best_child = None
        for child in self.children:
            exploitation = child.score / child.visits               # "How good has this node been so far?"
            exploration = exp * math.sqrt(                          # Encourages trying less-visited nodes.
                math.log(self.visits)/child.visits)
            uct_score = exploitation + exploration                  # UCT formula balances exploration vs exploitation.
            if uct_score > best_score:
                best_score = uct_score
                best_child = child
        return best_child

    def expand(self):                                               # Expand the tree by applying one untried move.
        move = self.untried_moves.pop()                             # Take one unexplored move.
        new_state = self.state.clone()                              # Clone current state and apply the move.
        new_state.apply_move(move, printing=False)
        child = MCTSNode(new_state, parent=self, move=move)         # Create new child node.
        self.children.append(child)
        return child

    def update(self, result):                                       # Update node statistics after a simulation.
        self.visits += 1                                            # Increment visit count.
        self.score += result                                        # Add simulation result (win/loss/draw score).

#### **4.2. Monte Carlo with Move Memory** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">Building on the standard version of MCTS implementation, we developed a second tree search version that addresses a specific problem in Pop Out: because of the pop move, the same board state can be reached multiple times during a game, which can cause the algorithm to waste simulations exploring cycles. 
<p style="text-indent: 2em;">In order to solve this, this next version makes each node remember all the states that already have appeared on the path from the root to that node. When expanding, it only considers moves that lead to states not yet seen on that path. During the simulation phase (rollout), the same principle is applied — repeated states are avoided whenever possible, and only used as a fallback if no other moves are available.
<p style="text-indent: 2em;">This avoids getting stuck in loops and produces more varied and realistic simulations. The trade-off is a small increase in computation per node, since visited states must be tracked and checked every time.</div>

In [3]:
def state_key(state):
    return state.board_to_string() + "TURN:" + state.cur_player     # A state should include both the board and the current player.

class MemoryNode(MCTSNode):
    def __init__(self, state, parent=None, move=None, visited_states=None):
        super().__init__(state, parent, move)
        if visited_states is None:
            self.visited_states = {state_key(state)}                # Store all states already visited in this path.
        else:
            self.visited_states = set(visited_states)
            self.visited_states.add(state_key(state))
        self.untried_moves = self.non_repeating_moves()             # Only keep moves that do not create a repeated state.

    def non_repeating_moves(self):
        non_repeating_moves = []
        for move in self.state.get_valid_moves():
            new_state = self.state.clone()
            new_state.apply_move(move, printing=False)
            if state_key(new_state) not in self.visited_states: non_repeating_moves.append(move)
        return non_repeating_moves
    
    def expand(self):
        move = self.untried_moves.pop()                             # Expand the tree by applying one untried move.
        state2 = self.state.clone()                                 # Take one unexplored move.
        state2.apply_move(move, printing=False)                     # Clone current state and apply the move.
        child = MemoryNode(state2, self, move, self.visited_states) # Create new child node.
        self.children.append(child)
        return child

#### **4.3. Monte Carlo with Heuristic Function** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">The third version of MCTS keeps the same tree structure as the standard one but replaces the random rollout with a heuristic-guided simulation, similar to what we have done in the second version of flat monte carlo. To chose the next move, it uses a simple heuristic: 

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**1. Win Immediately** *If there is a move that wins the game right away, take it;*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**2. Block the opponent** *Avoid moves that allow the opponent to win on the next turn;*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**3. Otherwise...** *If no such situations are detected, play randomly.*

<p style="text-indent: 2em;">The motivation behind this version is that random rollouts can be very misleading — a position that is clearly winning can look bad if the simulation plays randomly from it. By using smarter rollouts, the algorithm produces more realistic simulations and therefore better estimates of each move's value. The trade-off is that each simulation takes slightly longer, since the heuristic requires checking all valid moves at each step.</div>

In [4]:
def heuristic_move(state, valid_moves):                             # Selects options that produce immediate win, and avoids immediate losses.
    safe_moves = []                                                 # List to store moves that do not immediately lose the game.
    for move in valid_moves:                                        # Iterate through all possible valid moves.
        clone = state.clone()
        clone.apply_move(move, printing=False)
        if clone.winner == state.cur_player: return move            # If this move wins immediately, play it.
        elif clone.terminal: safe_moves.append(move)
        else:
            opponent_wins = False                                   # Check whether the opponent can win immediately after this move.
            for opp_move in clone.get_valid_moves():
                temp = clone.clone()
                temp.apply_move(opp_move, printing=False)
                if temp.winner == clone.cur_player:                 # If the opponent can win, mark it as unsafe.
                    opponent_wins = True
                    break
                if not opponent_wins: safe_moves.append(move)       # If the opponent cannot win immediately, consider this move safe.
    # If there are safe moves available, choose one randomly. If no safe moves exist, fall back to a completely random move.
    return random.choice(safe_moves) if safe_moves else random.choice(valid_moves)

#### **4.4. Monte Carlo with Move Memory AND Heuristic Function** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">Lastly, we have also made possible to combine the strenghts of Memory MCTS (4.2) and Heuristic MCTS (4.3), by defining a new version: Memory + Heuristic MCTS. This can be applied by feeding the heuristic function with the non-repeating moves instead of the full set of valid moves.
<p style="text-indent: 2em;">Below, we define the last two functions of our MCTS implementation, which are responsible for coordinating the entire search. The "rollout" function applies a new move to a simulation having in mind the type of search (random, heuristic, memory, memory+heuristic). And the "mcts_move" function returns the best move calculated for the current state of the game.</div>

In [5]:
def rollout(state, mode):                                           # Perform a simulation (playout) from a given state.
    simulation = state.clone()
    visited_states = {state_key(simulation)}
    while not simulation.terminal:                                  # Play random / with heuristic until the game ends.
        if mode == "random": move = random.choice(simulation.get_valid_moves())
        elif mode in ["memory", "memory+heuristic"]:
            nonrepeat_moves = []
            for move in simulation.get_valid_moves():
                temp = simulation.clone()
                temp.apply_move(move, printing=False)
                if state_key(temp) not in visited_states: nonrepeat_moves.append(move)
            if nonrepeat_moves: move = random.choice(nonrepeat_moves)# Prefer non-repeating moves.
            else: move = random.choice(simulation.get_valid_moves()) # Fallback: if all moves repeat, choose any valid move.
        elif mode == "heuristic": move = heuristic_move(simulation)
        simulation.apply_move(move, printing=False)
        visited_states.add(state_key(simulation))
    return simulation

def mcts_move(state, iterations, mode):                             # Main MCTS function: returns the best move from the current state.
    root = MCTSNode(state.clone())
    for _ in range(iterations):
        node = root
        if node.fully_expanded():                                   # SELECTION: Traverse the tree by selecting best children until we reach a leaf.
            if node.best_child(): node = node.best_child()
        if node.untried_moves:                                      # EXPANSION: If we haven't reached a terminal state, expand one new child.
            if not node.state.terminal: node = node.expand()
        final_state = rollout(node.state, mode)                     # SIMULATION: Play randomly from this node until the game ends.
        while node:                                                 # BACKPROPAGAITON: Propagate the result back up the tree.
            result = final_state.get_result(node.move_player)       # Get result from perspective of the player who just moved.
            node.update(result)
            node = node.parent                                      # Move up the tree.
    if not root.children: 
        return random.choice(state.get_valid_moves())               # In second version: If no child was created, fallback to a legal move.
    best_child = max(root.children, key=lambda child: child.visits) # Choose the most visited child as the final move.
    return best_child.move